# Week 59, Day 5: Case study: circular analysis and double dipping

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w059/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Inflation by selection

Simulate 400 neurons on 20 trials of each condition; 40 neurons have a true preference of 0.5 for A. Select neurons whose t-statistic for A against B exceeds 2, and report their mean A minus B on the same trials. Then select on trials 1 to 10 only and measure on trials 11 to 20.

In [ ]:
import numpy as np
gd = np.random.default_rng(595)
true_eff = np.where(np.arange(400) < 40, 0.5, 0.0)
A = true_eff[:, None] + gd.normal(size=(400, 20))
B = gd.normal(size=(400, 20))

def tstat(a, b):
    return (a.mean(1) - b.mean(1)) / np.sqrt(a.var(1, ddof=1) / a.shape[1] + b.var(1, ddof=1) / b.shape[1])

sel = tstat(A, B) > 2
circular = np.mean(A[sel].mean(1) - B[sel].mean(1))
truth_sel = true_eff[sel].mean()

sel_half = tstat(A[:, :10], B[:, :10]) > 2
held_out = ...
truth_half = true_eff[sel_half].mean()
print(f"selected {sel.sum()} neurons; circular estimate {circular:.2f}, their true mean effect {truth_sel:.2f}")
print(f"selected on half: {sel_half.sum()} neurons; held-out estimate {held_out:.2f}, their true mean effect {truth_half:.2f}")

In [ ]:
import numpy as np
def test_circular():
    "the circular estimate is more than 1.5 times the true effect of the selected neurons"
    assert circular > 1.5 * truth_sel
def test_split():
    "the held-out estimate is within 0.1 of the true effect of the neurons selected on the first half"
    assert abs(held_out - truth_half) < 0.1

test_circular()
test_split()
print("All checks passed")

### Exercise: Neurons that seem to adapt

Simulate 300 neurons with stable true responses, measured with noise on two days. Select the top 10% and the bottom 10% on day 1, and compare their mean responses on the two days.

In [ ]:
gr = np.random.default_rng(5951)
true_resp = gr.normal(5, 1, 300)
day1 = true_resp + gr.normal(0, 1, 300)
day2 = true_resp + gr.normal(0, 1, 300)
top = day1 >= np.percentile(day1, 90)
bottom = ...
day1_top, day2_top = day1[top].mean(), day2[top].mean()
day1_bottom, day2_bottom = day1[bottom].mean(), day2[bottom].mean()
print(f"top 10% on day 1:    {day1_top:.2f} then {day2_top:.2f}")
print(f"bottom 10% on day 1: {day1_bottom:.2f} then {day2_bottom:.2f}")

In [ ]:
import numpy as np
def test_drop():
    "the top neurons on day 1 respond less on day 2, and the bottom ones more, with no real change"
    assert day2_top < day1_top - 0.3 and day2_bottom > day1_bottom + 0.3

test_drop()
print("All checks passed")

### Exercise: Repair the pipeline

Write honest_pipeline(X, y): use three folds, each made of two consecutive whole blocks (one of each condition), and inside each fold compute the z-scoring means and standard deviations and the feature selection from training trials only.

In [ ]:
ga = np.random.default_rng(5952)
T = 120
y_a = (np.arange(T) // 20) % 2
X_a = np.cumsum(0.25 * ga.normal(size=(T, 500)), axis=0) + ga.normal(size=(T, 500))

def select(X, y, k=20):
    return np.argsort(np.abs(X[y == 1].mean(0) - X[y == 0].mean(0)))[-k:]

def centroid_score(Xtr, ytr, Xte, yte):
    c0, c1 = Xtr[ytr == 0].mean(0), Xtr[ytr == 1].mean(0)
    return np.sum((((Xte - c1) ** 2).sum(1) < ((Xte - c0) ** 2).sum(1)) == yte)

def published_pipeline(X, y):
    Z = (X - X.mean(0)) / X.std(0)                         # leak 1: statistics from all trials
    f = select(Z, y)                                       # leak 2: selection on all trials
    folds = np.array_split(np.random.default_rng(0).permutation(len(y)), 5)   # leak 3: random folds
    right = 0
    for te in folds:
        tr = np.setdiff1d(np.arange(len(y)), te)
        right += centroid_score(Z[tr][:, f], y[tr], Z[te][:, f], y[te])
    return right / len(y)

def honest_pipeline(X, y):
    folds = np.array_split(np.arange(len(y)), 3)           # pairs of whole blocks, one of each condition
    right = 0
    for te in folds:
        tr = np.setdiff1d(np.arange(len(y)), te)
        ...
    return right / len(y)

leaky_acc, honest_acc = published_pipeline(X_a, y_a), honest_pipeline(X_a, y_a)
print(f"published pipeline: {leaky_acc:.3f}; honest pipeline: {honest_acc:.3f}")

In [ ]:
import numpy as np
def test_leaky():
    "the published pipeline reports well above chance on null data"
    assert leaky_acc > 0.7
def test_honest():
    "the honest pipeline reports accuracy near chance (0.35 to 0.65)"
    assert 0.35 < honest_acc < 0.65

test_leaky()
test_honest()
print("All checks passed")